In [2]:
import os
import sys
from pathlib import Path
import pandas_ta as pta
import numpy as np

# 1. Բոտին հասկացնում ենք, թե որտեղ է գտնվում մեր հիմնական freqtrade պանակը
# Քանի որ մենք user_data/notebooks-ում ենք, երկու քայլ հետ ենք գնում
project_root = Path(os.getcwd()).parent.parent
sys.path.append(str(project_root))

# 2. Ներմուծում ենք Freqtrade-ի տվյալների բեռնման գործիքը
from freqtrade.data.history import load_pair_history
from freqtrade.resolvers import StrategyResolver

print("Միջավայրը հաջողությամբ պատրաստ է:")

Միջավայրը հաջողությամբ պատրաստ է:


In [3]:
from freqtrade.enums import CandleType

# 1. Հասցեն ուղղում ենք դեպի բուն binance պանակը
real_user_data_dir = project_root / 'user_data'

config = {
    'strategy': 'MySupertrand_V6',
    'user_data_dir': real_user_data_dir,
    'trading_mode': 'futures'
}

# Կանչում ենք ստրատեգիան
strategy = StrategyResolver.load_strategy(config)

# 2. Բեռնում ենք ՖՅՈՒՉԵՐՍԱՅԻՆ BTC տվյալները feather ֆորմատով
dataframe = load_pair_history(
    datadir=real_user_data_dir / 'data' / 'binance',
    timeframe=strategy.timeframe,
    pair='BTC/USDT:USDT',
    data_format='feather',            # 🔥 Փոխեցինք feather-ի
    candle_type=CandleType.FUTURES
)

print(f"Հաջողությա՛մբ բեռնվեց: Աղյուսակում կա {len(dataframe)} ՖՅՈՒՉԵՐՍԱՅԻՆ մոմ:")

Հաջողությա՛մբ բեռնվեց: Աղյուսակում կա 81642 ՖՅՈՒՉԵՐՍԱՅԻՆ մոմ:


In [4]:
# 1. Աշխատեցնում ենք քո ռազմավարության populate_indicators ֆունկցիան
dataframe = strategy.populate_indicators(dataframe, {'pair': 'BTC/USDT:USDT'})

# 2. Էկրանին ենք հանում աղյուսակի վերջին 10 տողերը՝ տեսնելու համար նոր սյունակները
dataframe.tail(10)

,date,open,high,low,close,volume,st_line,st_dir
81632,2026-04-30 08:00:00+00:00,76129.9,76180.0,76036.7,76054.9,1318.911,75602.945756,1
81633,2026-04-30 08:15:00+00:00,76054.9,76132.3,75987.6,76045.3,2454.947,75602.945756,1
81634,2026-04-30 08:30:00+00:00,76045.4,76107.7,76012.0,76104.1,459.769,75602.945756,1
81635,2026-04-30 08:45:00+00:00,76104.2,76116.5,76020.4,76025.7,458.590,75610.179206,1
81636,2026-04-30 09:00:00+00:00,76025.8,76076.8,75989.0,76050.8,591.297,75610.179206,1
81637,2026-04-30 09:15:00+00:00,76050.8,76197.2,75973.3,76196.1,619.192,75623.174657,1
81638,2026-04-30 09:30:00+00:00,76196.0,76332.6,76035.8,76042.0,1741.493,75679.292191,1
81639,2026-04-30 09:45:00+00:00,76042.0,76168.0,76035.1,76062.7,666.217,75679.292191,1
81640,2026-04-30 10:00:00+00:00,76062.7,76136.6,76004.0,76014.4,479.655,75679.292191,1
81641,2026-04-30 10:15:00+00:00,76014.3,76046.1,75943.3,75952.8,887.561,75679.292191,1


In [5]:
# 1. Հաշվում ենք հիմնական EMA 32-ը
dataframe['ema'] = pta.ema(dataframe['close'], length=32)

# 2. Հաշվում ենք ATR-ը (14)
dataframe['atr'] = pta.atr(dataframe['high'], dataframe['low'], dataframe['close'], length=14)

# 3. Ստեղծում ենք ճշգրիտ թվային զոնան EMA-ի շուրջ (± 0.5 * ATR)
dataframe['ema_upper'] = dataframe['ema'] + (dataframe['atr'] * 0.5)
dataframe['ema_lower'] = dataframe['ema'] - (dataframe['atr'] * 0.5)

# 4. Հաշվում ենք EMA-ի Գծային Ռեգրեսիայի Slope-ը (5 մոմ հետ նայելով)
dataframe['ema_slope'] = pta.slope(dataframe['ema'], length=5)

# 5. Ստեղծում ենք ֆիլտրը. 1, եթե slope-ը մեծ է 0-ից (բարձրացող է), և 0, եթե ոչ
dataframe['ema_is_rising'] = np.where(dataframe['ema_slope'] > 0, 1, 0)
# dataframe['ema_is_rising'] = np.where(
#     (dataframe['ema_slope'] > 0) &
#     (dataframe['ema_slope'] > dataframe['ema_slope'].shift(1)),
#     1, 0
# )
# Տեսնենք հաշվարկված թվերը
dataframe[['close', 'ema', 'ema_upper', 'ema_lower', 'ema_slope', 'ema_is_rising']].tail(5)

,close,ema,ema_upper,ema_lower,ema_slope,ema_is_rising
81637,76196.1,75889.735348,75971.930676,75807.540021,14.523079,1
81638,76042.0,75898.963509,75985.887742,75812.039276,13.602893,1
81639,76062.7,75908.886933,75994.348721,75823.425145,12.276657,1
81640,76014.4,75915.281664,75999.374753,75831.188576,11.395647,1
81641,75952.8,75917.555503,75999.313371,75835.797635,9.517123,1


In [6]:
# 1. Ֆիքսում ենք LONG Հետցատկի մոմը (Bounce)` Ռեգրեսիայի ֆիլտրով և Զոնայի թվերով
# dataframe['ema_bounce_long'] = np.where(
#     (dataframe['st_dir'] == 1) &
#     (dataframe['ema_is_rising'] == 1) &            # EMA-ն պետք է ընդհանուր աճող լինի (Linear Regression)
#     (dataframe['low'] <= dataframe['ema_upper']) &  # Գնի նվազագույնը մտել է զոնայի մեջ
#     (dataframe['close'] > dataframe['ema_lower']),   # Բայց մոմը փակվել է ստորին սահմանից վերև
#     True, False
# )
dataframe['ema_bounce_long'] = np.where(
    (dataframe['st_dir'] == 1) &
    (dataframe['ema_is_rising'] == 1) &            # EMA-ն պետք է ընդհանուր աճող լինի (Linear Regression)
    (dataframe['low'] <= dataframe['ema']) &  # Գնի նվազագույնը մտել է զոնայի մեջ
    (dataframe['close'] > dataframe['ema']),   # Բայց մոմը փակվել է ստորին սահմանից վերև
    True, False
)

# 2. Վերջնական Մուտքը (enter_long)՝ հաջորդ մոմի հաստատմամբ
dataframe['enter_long'] = 0
dataframe.loc[
    (dataframe['st_dir'] == 1) &
    (dataframe['ema_is_rising'] == 1) &
    (dataframe['ema_bounce_long'].shift(1) == True) & # Նախորդ մոմը եղել է զոնայի bounce
    (dataframe['close'] > dataframe['ema']),     # Ընթացիկ մոմը հաստատել է՝ փակվելով զոնայից վերև
    'enter_long'
] = 1

# Տեսնենք միայն այն 5 պահերը, երբ բոտը LONG մուտքի հրաման է տվել
dataframe[dataframe['enter_long'] == 1][['close', 'ema', 'ema_is_rising', 'enter_long']].tail(5)

,close,ema,ema_is_rising,enter_long
81295,78183.1,78045.779397,1,1
81304,78521.9,78122.468494,1,1
81307,78274.7,78157.838465,1,1
81311,78613.5,78213.595679,1,1
81511,76534.9,76292.567442,1,1


In [7]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Վերցնում ենք վերջին 500 մոմերը
df_plot = dataframe.iloc[-2000:].copy()

# 2. Առանձնացնում ենք Supertrend-ի գծերը
df_plot['st_up'] = np.where(df_plot['st_dir'] == 1, df_plot['st_line'], np.nan)
df_plot['st_down'] = np.where(df_plot['st_dir'] == -1, df_plot['st_line'], np.nan)

# 3. Առանձնացնում ենք ազդանշանները
long_signals = df_plot[df_plot['enter_long'] == 1]

# 4. 🔥 Ստեղծում ենք 2 հարկանի գրաֆիկ (Հարկ 1-ը՝ Գին, Հարկ 2-ը՝ Slope)
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,          # Իրար հետ համակցված X առանցք (ժամանակ)
    vertical_spacing=0.05,       # Հարկերի արանքի հեռավորությունը
    row_heights=[0.75, 0.25]    # Գնի գրաֆիկը կզբաղեցնի 75%, իսկ slope-ը՝ 25%
)

# ================== ՀԱՐԿ 1. ԳՆԻ ԳՐԱՖԻԿ ==================

# Ճապոնական մոմերը
fig.add_trace(go.Candlestick(
    x=df_plot['date'], open=df_plot['open'], high=df_plot['high'],
    low=df_plot['low'], close=df_plot['close'], name='Գին (BTC)'
), row=1, col=1)

# Supertrend Աճ / Անկում
fig.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['st_up'], mode='lines', name='Supertrend Աճ', line=dict(color='#00FF00', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['st_down'], mode='lines', name='Supertrend Անկում', line=dict(color='#FF0000', width=1.5)), row=1, col=1)

# EMA Զոնան (Վերևի/Ներքևի սահմաններով և լուսավորումով)
fig.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['ema_upper'], mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'), row=1, col=1)
fig.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['ema_lower'], mode='lines', line=dict(width=0), fill='tonexty', fillcolor='rgba(0, 255, 255, 0.12)', name='EMA ATR Զոնա'), row=1, col=1)
fig.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['ema'], mode='lines', name='EMA 32', line=dict(color='rgba(0, 255, 255, 0.5)', width=1, dash='dot')), row=1, col=1)

# LONG Մուտքի Կանաչ Սլաքները
fig.add_trace(go.Scatter(
    x=long_signals['date'], y=long_signals['low'] * 0.998,
    mode='markers', name='EMA Bounce LONG',
    marker=dict(symbol='triangle-up', size=13, color='#00FF00', line=dict(width=1, color='white'))
), row=1, col=1)


# ================== ՀԱՐԿ 2. EMA SLOPE ԳՐԱՖԻԿ ==================

# 🔥 Նկարում ենք Գծային Ռեգրեսիայի Slope-ի գիծը (Դեղին գույնով)
fig.add_trace(go.Scatter(
    x=df_plot['date'], y=df_plot['ema_slope'],
    mode='lines', name='EMA Linear Regression Slope',
    line=dict(color='#FFD700', width=2)
), row=2, col=1)

# 🔥 Ավելացնում ենք 0-ի հորիզոնական առանցքը (Մոխրագույն կտրտվող գիծ)
fig.add_trace(go.Scatter(
    x=df_plot['date'], y=[0]*len(df_plot),
    mode='lines', name='0-ի Սահմանագիծ',
    line=dict(color='#666666', width=1, dash='dash'),
    showlegend=False
), row=2, col=1)


# ================== ՁԵՎԱՎՈՐՈՒՄ ԵՎ ԽԱՉԱՁԵՎ ԿՈՒՐՍՈՐ ==================

# Խաչաձև Կուրսոր X և Y առանցքների համար (Կաշխատի երկու հարկերում էլ միաժամանակ)
fig.update_xaxes(showspikes=True, spikemode='across', spikesnap='cursor', spikedash='dash', spikecolor='#aaaaaa', spikethickness=1)
fig.update_yaxes(showspikes=True, spikemode='across', spikesnap='cursor', spikedash='dash', spikecolor='#aaaaaa', spikethickness=1)

fig.update_layout(
    title='BTC/USDT + EMA ATR Զոնա + Linear Regression Slope (V6.2)',
    yaxis_title='Գին (USDT)', yaxis2_title='Slope Արժեք', xaxis2_title='Ամսաթիվ',
    template='plotly_dark', xaxis_rangeslider_visible=False,
    dragmode='pan', hovermode='closest',
    height=900, # Կարող ես մի փոքր իջեցնել, որ ավելի լավ տեղավորվի էկրանիդ

    # 🔥 ՈՒՂՂՈՒՄ. Այս երկու տողերը կվերացնեն սպիտակ ֆոնը
    plot_bgcolor='rgb(17, 17, 17)',   # Գրաֆիկի ներսի ֆոնը
    paper_bgcolor='rgb(17, 17, 17)'   # Ամբողջ թղթի/էջի արտաքին ֆոնը
)

# Ցուցադրում՝ մկնիկի անիվով Zoom անելով
fig.show(config={'scrollZoom': True}, renderer='browser')